# 🎵 Huấn luyện Mô hình Music Emotion Recognition (MERT-v1-95M) trên Google Colab

> **Dự án:** SE121-microservices (Hệ thống Mạng xã hội Tích hợp AI Chăm sóc Sức khỏe Tinh thần)  
> **Mục tiêu:** Fine-tune mô hình Music Foundation Transformer (`m-a-p/MERT-v1-95M` - ICLR 2024) để dự đoán chính xác tọa độ cảm xúc 2D **Valence** và **Arousal** (thang đo $[0.0, 1.0]$), sau đó xuất sang định dạng **ONNX INT8** để triển khai trên CPU Server.  
> **Cấu hình khuyên dùng:** Google Colab GPU T4 (Miễn phí).

### 1. Cài đặt Môi trường & Thư viện Cần thiết

In [ ]:
# Cài đặt các thư viện Deep Learning & Audio SOTA
!pip install -q transformers torchaudio soundfile accelerate onnx onnxruntime tqdm pandas scikit-learn scipy

import os
import gc
import torch
import torchaudio
import numpy as np
import pandas as pd

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

### 2. Kết nối Google Drive & Giải nén Dữ liệu Audio

In [ ]:
import zipfile
from google.colab import drive

# 1. Mount Google Drive
try:
    drive.mount('/content/drive')
except Exception:
    pass

AUDIO_DIR = '/content/audio'
ZIP_CANDIDATES = [
    '/content/drive/MyDrive/Dataset/audio.zip',
    '/content/drive/MyDrive/audio.zip',
    '/content/drive/MyDrive/music/audio.zip',
    '/content/audio.zip'
]

# 2. Tìm file zip
found_zip = None
for z in ZIP_CANDIDATES:
    if os.path.exists(z):
        found_zip = z
        break

# 3. Giải nén audio nếu chưa có hoặc thư mục đang rỗng
existing_mp3 = len([f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.mp3')]) if os.path.exists(AUDIO_DIR) else 0

if existing_mp3 == 0:
    if found_zip:
        print(f"📦 Tìm thấy file zip tại: {found_zip}")
        print("Đang giải nén vào /content/audio...")
        !rm -rf /content/audio
        !unzip -q -o "{found_zip}" -d /content/
        if not os.path.exists(AUDIO_DIR) and any(f.endswith('.mp3') for f in os.listdir('/content')):
            !mkdir -p /content/audio && mv /content/*.mp3 /content/audio/
    else:
        print("⚠️ Chưa tìm thấy audio.zip! Vui lòng kiểm tra đường dẫn trên Google Drive.")

total_mp3 = len([f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.mp3')]) if os.path.exists(AUDIO_DIR) else 0
print(f"✓ Tổng số file MP3 sẵn sàng: {total_mp3} file!")

# 4. Đọc dataset splits
def find_csv(name):
    paths = [
        f'/content/drive/MyDrive/Dataset/{name}',
        f'/content/drive/MyDrive/{name}',
        f'/content/drive/MyDrive/music/{name}',
        f'/content/{name}'
    ]
    for p in paths:
        if os.path.exists(p):
            return p
    return name

train_df = pd.read_csv(find_csv('train.csv'))
val_df = pd.read_csv(find_csv('val.csv'))
test_df = pd.read_csv(find_csv('test.csv'))

print(f"✓ Đã nạp thành công Dataset splits:")
print(f"  • Train set: {len(train_df)} bài")
print(f"  • Val set  : {len(val_df)} bài")
print(f"  • Test set : {len(test_df)} bài")

### 3. Chuẩn bị PyTorch Dataset Class & Trích xuất Âm thanh 24kHz (15s Thin-Slicing)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T
import torch.nn.functional as F

TARGET_SR = 24000  # MERT chuẩn hóa âm thanh ở 24,000 Hz
DURATION_SEC = 15  # Chuẩn khoa học Thin-slicing (15s điệp khúc - Tối ưu 4x VRAM cho GPU T4)
TARGET_SAMPLES = TARGET_SR * DURATION_SEC

class MusicEmotionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, audio_dir: str = '/content/audio', is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.is_train = is_train
        self.resampler_cache = {}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        valence = float(row["valence"])
        arousal = float(row["arousal"])
        target = torch.tensor([valence, arousal], dtype=torch.float32)

        # Tìm file audio theo nhiều mẫu tên: audio_filename, track_id, raw_id
        audio_candidates = []
        if "audio_filename" in row:
            audio_candidates.append(str(row["audio_filename"]))
        if "track_id" in row:
            audio_candidates.append(f"{row['track_id']}.mp3")
        if "raw_id" in row:
            audio_candidates.append(f"{row['raw_id']}.mp3")

        audio_file = None
        for name in audio_candidates:
            p = os.path.join(self.audio_dir, name)
            if os.path.exists(p):
                audio_file = p
                break

        if audio_file and os.path.exists(audio_file):
            try:
                waveform, sr = torchaudio.load(audio_file)
                # 1. Đổi sang Mono nếu là Stereo
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)
                # 2. Resample sang 24kHz
                if sr != TARGET_SR:
                    if sr not in self.resampler_cache:
                        self.resampler_cache[sr] = T.Resample(sr, TARGET_SR)
                    waveform = self.resampler_cache[sr](waveform)
                
                # 3. Center-crop 15s (Đoạn điệp khúc/trọng tâm cảm xúc)
                total_samples = waveform.shape[1]
                if total_samples > TARGET_SAMPLES:
                    start = (total_samples - TARGET_SAMPLES) // 2
                    waveform = waveform[:, start:start + TARGET_SAMPLES]
                elif total_samples < TARGET_SAMPLES:
                    waveform = F.pad(waveform, (0, TARGET_SAMPLES - total_samples))
                
                audio_tensor = waveform.squeeze(0)
            except Exception as e:
                audio_tensor = torch.zeros(TARGET_SAMPLES, dtype=torch.float32)
        else:
            audio_tensor = torch.zeros(TARGET_SAMPLES, dtype=torch.float32)

        return audio_tensor, target

print("✓ Lớp Dataset 15s Thin-slicing sẵn sàng!")

### 4. Kiến trúc Mô hình: MERT-v1-95M + Temporal Attention Pooling

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class MERTMusicEmotionModel(nn.Module):
    def __init__(self, model_name="m-a-p/MERT-v1-95M", freeze_layers=8):
        super().__init__()
        print(f"Đang nạp Backbone Transformer: {model_name}...")
        self.mert = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        
        # Freeze Conv Feature Extractor & 8 tầng Transformer đầu để bảo tồn tri thức âm nhạc gốc
        if freeze_layers > 0:
            for param in self.mert.feature_extractor.parameters():
                param.requires_grad = False
            for layer in self.mert.encoder.layers[:freeze_layers]:
                for param in layer.parameters():
                    param.requires_grad = False
                    
        hidden_size = self.mert.config.hidden_size # 768
        
        # 1. Temporal Attention Pooling (Tự động phát hiện đoạn cao trào / Chorus)
        self.attention_pool = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # 2. Multi-Head Regressor cho Valence & Arousal
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 2),
            nn.Sigmoid()  # Ép giá trị đầu ra về miền [0.0, 1.0]
        )

    def forward(self, input_values):
        # input_values: [Batch, Samples]
        outputs = self.mert(input_values)
        hidden_states = outputs.last_hidden_state # [Batch, Frames, 768]
        
        # Attention Weights theo trục thời gian
        att_weights = torch.softmax(self.attention_pool(hidden_states), dim=1)
        context = torch.sum(att_weights * hidden_states, dim=1) # [Batch, 768]
        
        out = self.regressor(context) # [Batch, 2]
        return out

print("✓ Kiến trúc mô hình MERT Music Emotion sẵn sàng!")

### 5. Hàm Mất Mát: Concordance Correlation Coefficient (CCC Loss)

In [ ]:
class CCCLoss(nn.Module):
    """
    Lin's Concordance Correlation Coefficient (CCC) Loss
    Chuẩn vàng đo lường độ tương quan liên tục trong hồi quy cảm xúc (Valence/Arousal).
    """
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, y_pred, y_true):
        loss = 0.0
        for i in range(y_pred.shape[1]):
            x = y_pred[:, i]
            y = y_true[:, i]
            vx = x - torch.mean(x)
            vy = y - torch.mean(y)
            rho = torch.sum(vx * vy) / (torch.sqrt(torch.sum(vx ** 2)) * torch.sqrt(torch.sum(vy ** 2)) + self.eps)
            x_m, y_m = torch.mean(x), torch.mean(y)
            x_s, y_s = torch.var(x, unbiased=False), torch.var(y, unbiased=False)
            ccc = (2.0 * rho * torch.sqrt(x_s) * torch.sqrt(y_s)) / (x_s + y_s + (x_m - y_m)**2 + self.eps)
            loss += (1.0 - ccc)
        return loss / y_pred.shape[1]

class CombinedEmotionLoss(nn.Module):
    def __init__(self, alpha=0.75):
        super().__init__()
        self.alpha = alpha
        self.ccc = CCCLoss()
        self.mse = nn.MSELoss()

    def forward(self, y_pred, y_true):
        return self.alpha * self.ccc(y_pred, y_true) + (1.0 - self.alpha) * self.mse(y_pred, y_true)

### 6. Vòng Lặp Huấn Luyện (Training Loop) với Mixed Precision (AMP)

In [ ]:
import gc
from tqdm import tqdm

# 1. Dọn dẹp GPU cache trước khi bắt đầu
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MERTMusicEmotionModel(freeze_layers=8).to(device)
criterion = CombinedEmotionLoss(alpha=0.75)

# Optimizer & Scheduler
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

# DataLoader (Batch size 4 tối ưu mượt mà cho GPU T4)
BATCH_SIZE = 4
train_loader = DataLoader(MusicEmotionDataset(train_df, AUDIO_DIR, is_train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(MusicEmotionDataset(val_df, AUDIO_DIR, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

EPOCHS = 15
best_val_loss = float('inf')
MODEL_SAVE_PATH = 'best_mert_emotion.pt'

print(f"🚀 Bắt đầu huấn luyện MERT trên: {device} ({EPOCHS} Epochs, Batch={BATCH_SIZE})...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for audio, targets in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS:02d}"):
        audio, targets = audio.to(device), targets.to(device)
        optimizer.zero_grad()
        
        if torch.cuda.is_available():
            with torch.amp.autocast('cuda'):
                preds = model(audio)
                loss = criterion(preds, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(audio)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            
        train_loss += loss.item()
    
    scheduler.step()
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for audio, targets in val_loader:
            audio, targets = audio.to(device), targets.to(device)
            if torch.cuda.is_available():
                with torch.amp.autocast('cuda'):
                    preds = model(audio)
                    val_loss += criterion(preds, targets).item()
            else:
                preds = model(audio)
                val_loss += criterion(preds, targets).item()
                
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        # Tự động backup vào Google Drive nếu có
        for drive_bk in ['/content/drive/MyDrive/Dataset/best_mert_emotion.pt', '/content/drive/MyDrive/best_mert_emotion.pt']:
            try:
                torch.save(model.state_dict(), drive_bk)
                break
            except Exception:
                pass
        print(f"  ⭐ Đã lưu Best Model Checkpoint tại Epoch {epoch+1} (Val Loss: {best_val_loss:.4f})")

print("🎉 HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")

### 7. Đánh giá Nghiệm thu trên Tập Held-Out Test Set

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

def calc_ccc_np(y_true, y_pred):
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true, var_pred = np.var(y_true), np.var(y_pred)
    covar = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    return float((2 * covar) / (var_true + var_pred + (mean_true - mean_pred)**2 + 1e-8))

# Nạp checkpoint tốt nhất
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()

test_loader = DataLoader(MusicEmotionDataset(test_df, AUDIO_DIR, is_train=False), batch_size=4, shuffle=False)

all_preds, all_targets = [], []
with torch.no_grad():
    for audio, targets in tqdm(test_loader, desc="Đánh giá Test Set"):
        audio = audio.to(device)
        preds = model(audio)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

v_true, a_true = all_targets[:, 0], all_targets[:, 1]
v_pred, a_pred = all_preds[:, 0], all_preds[:, 1]

print("\n" + "=" * 70)
print("           KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP HELD-OUT TEST SET (386 BÀI)")
print("=" * 70)
print(f"Valence R2 Score : {r2_score(v_true, v_pred):.4f}")
print(f"Valence MAE      : {mean_absolute_error(v_true, v_pred):.4f}")
print(f"Valence CCC      : {calc_ccc_np(v_true, v_pred):.4f}")
print("-" * 70)
print(f"Arousal R2 Score : {r2_score(a_true, a_pred):.4f}")
print(f"Arousal MAE      : {mean_absolute_error(a_true, a_pred):.4f}")
print(f"Arousal CCC      : {calc_ccc_np(a_true, a_pred):.4f}")
print("=" * 70 + "\n")

### 8. Đóng gói ONNX INT8 & Tự động Tải về Máy Server Backend

In [ ]:
  # 1. Cài đặt thêm onnxscript và onnxruntime                                                                    
  !pip install -q onnxscript onnx onnxruntime                                                                    
                                                                                                                 
  import os                                                                                                      
  import torch                                                                                                   
  import onnx                                                                                                    
  from onnxruntime.quantization import quantize_dynamic, QuantType                                               
  from google.colab import files                                                                                 
  import shutil                                                                                                  
                                                                                                                   
  # 2. Nạp mô hình vào CPU để export                                                                             
  model = MERTMusicEmotionModel(freeze_layers=8)                                                                 
  model.load_state_dict(torch.load("best_mert_emotion.pt", map_location="cpu"))                                  
  model.eval()                                                                                                   
  model_cpu = model.to("cpu")                                                                                    
                                                                                                                   
  dummy_input = torch.randn(1, 24000 * 15, device="cpu")                                                         
  fp32_onnx = "mert_emotion_fp32.onnx"                                                                           
  int8_onnx = "mert_emotion_int8.onnx"                                                                           
                                                                                                                   
  # 3. Xuất sang ONNX FP32                                                                                       
  print("🚀 Đang xuất mô hình sang ONNX FP32...")                                                                
  torch.onnx.export(                                                                                             
      model_cpu,                                                                                                 
      dummy_input,                                                                                               
      fp32_onnx,                                                                                                 
      input_names=["audio_waveform"],                                                                            
      output_names=["valence_arousal"],                                                                          
      dynamic_axes={                                                                                             
          "audio_waveform": {0: "batch_size", 1: "samples"},                                                     
          "valence_arousal": {0: "batch_size"}                                                                   
      },                                                                                                         
      opset_version=14,                                                                                          
      dynamo=False  # Sử dụng TorchScript ONNX Exporter ổn định nhất                                             
  )                                                                                                              
  print("✓ Xuất FP32 ONNX thành công!")                                                                          
                                                                                                                   
  # 4. Lượng tử hóa Dynamic INT8 Quantization (Giảm 75% dung lượng, tối ưu CPU Server)                           
  print("⚡ Đang lượng tử hóa sang ONNX INT8...")                                                                
  quantize_dynamic(                                                                                              
      model_input=fp32_onnx,                                                                                     
      model_output=int8_onnx,                                                                                    
      weight_type=QuantType.QUInt8                                                                               
  )                                                                                                              
  int8_size_mb = os.path.getsize(int8_onnx) / (1024 * 1024)                                                      
  print(f"✓ Xuất INT8 ONNX thành công! Kích thước siêu nhẹ: {int8_size_mb:.2f} MB")                              
                                                                                                                   
  # 5. Lưu bản sao dự phòng vào Google Drive                                                                     
  for drive_out in ['/content/drive/MyDrive/Dataset/mert_emotion_int8.onnx', '/content/drive/MyDrive/mert_emotion_int8.onnx']:                                                                
      try:                                                                                                       
          shutil.copyfile(int8_onnx, drive_out)                                                                  
          print(f"✓ Đã lưu bản backup vào Google Drive: {drive_out}")                                            
          break                                                                                                  
      except Exception:                                                                                          
          pass                                                                                                   
                                                                                                                   
  # 6. Tự động tải về máy tính của bạn                                                                           
  try:                                                                                                           
      print("⬇️ Đang bật popup tải file mô hình về máy tính...")                                                 
      files.download(int8_onnx)                                                                                  
  except Exception:                                                                                              
      print("Bạn có thể tải thủ công file mert_emotion_int8.onnx trong tab Files!")